In [ ]:
import json

with open('IMDB_top_100.json', 'r') as file:
    imdb_dataset = json.load(file)

In [ ]:
vespa_feed = []

for movie in imdb_dataset:
    vespa_feed.append({
        'id': movie['id'],
        'fields': {
            'title': movie['Series_Title'],
            'description': movie['Synopsis'],
            'id': movie['id'],
            'year': movie['Released_Year'],
        }   
    })

print(vespa_feed)



In [ ]:
from vespa.application import ApplicationPackage

app_package = ApplicationPackage(name='imdbsearch')

In [ ]:
from vespa.package import Schema, Document

film_schema = Schema(name='imdb_film_schema', document=Document())

In [ ]:
from vespa.package import Field

film_schema.add_fields(
    Field(
        name='title',
        type='string',
        indexing=['index', 'summary']
    ),
    Field(
        name='description',
        type='string',
        indexing=['index', 'summary']
    ),
    Field(
        name='year',
        type='int',
        indexing=['attribute']
    ),
    Field(
        name='id',
        type='string',
        indexing=['summary']
    )
)

In [ ]:
from vespa.package import FieldSet

film_schema.add_field_set(
    FieldSet(
        name='default',
        fields=['title', 'description', 'year', 'id']
    ))

In [ ]:
app_package.add_schema(film_schema)

In [ ]:
from vespa.deployment import VespaCloud

vespa_cloud = VespaCloud(
    tenant='akashsinghal',
    application='imdbsearch',
    application_package=app_package
)

In [ ]:
app = vespa_cloud.deploy(region="aws-euw1-az1")

In [ ]:
cert_path = app.cert
key_path = app.key

endpoint = vespa_cloud.get_mtls_endpoint()

In [ ]:
from vespa.application import Vespa

vespa_application = Vespa(endpoint, cert=cert_path, key=key_path)

In [ ]:
from vespa.io import VespaResponse

def callback(response: VespaResponse, id:str):
    if not response.is_successful():
        print(f"Error when feeding document {id}: {response.get_json()}")

vespa_application.feed_iterable(vespa_feed, schema='imdb_film_schema', callback=callback)

In [ ]:
from vespa.io import VespaQueryResponse

def print_hits(response: VespaQueryResponse):
    for i, hit in enumerate(response.hits):
        print(f'{i+1:>3}  {hit["fields"]['id']:>5}  {hit['fields']['title']}')

In [ ]:
# Searching for 'world war' using userQuery(). This function uses the default fieldset

response = vespa_application.query(
    yql="select * from sources * where default contains 'world war'", # The default limit in Vespa is 10 results
    # query = "world war", # when using userQuery() you must provide the query as a parameter
)

print_hits(response)


In [ ]:
# Searching for 'world war' using userQuery(). This function uses the default fieldset

response = vespa_application.query(
    yql="select * from sources * where userQuery()", # The default limit in Vespa is 10 results
    query = "world war", # when using userQuery() you must provide the query as a parameter
)

print_hits(response)
